# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
import re

load_dotenv(override=True)




DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
MODEL = "groq/meta-llama/llama-4-scout-17b-16e-instruct"

In [2]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]
    
    @classmethod
    def clean_and_parse(cls, json_str: str):
        """Clean common JSON issues from Groq responses"""
        # Remove any trailing commas before closing braces
        json_str = re.sub(r',\s*}', '}', json_str)
        json_str = re.sub(r',\s*]', ']', json_str)
        # Fix any extra closing braces
        json_str = re.sub(r'}\s*}', '}', json_str)
        return cls.model_validate_json(json_str)

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [6]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [7]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [10]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [11]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# A

In [29]:
def process_document(document):
    messages = make_messages(document)
    try:
        response = completion(model=MODEL, messages=messages, response_format=Chunks)
        reply = response.choices[0].message.content
        doc_as_chunks = Chunks.clean_and_parse(reply).chunks
        return [chunk.as_result(document) for chunk in doc_as_chunks]
    except Exception as e:
        print(f"Error processing document {document['source']}: {e}")
        # Fallback: try without response_format
        response = completion(model=MODEL, messages=messages)
        reply = response.choices[0].message.content
        # Try to extract JSON from the response
        json_match = re.search(r'\{.*\}', reply, re.DOTALL)
        if json_match:
            doc_as_chunks = Chunks.clean_and_parse(json_match.group()).chunks
            return [chunk.as_result(document) for chunk in doc_as_chunks]
        else:
            print(f"Could not extract JSON from response for {document['source']}")
            return []

In [20]:
process_document(documents[0])

[Result(page_content='Company Overview\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup.\n\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Rapid Growth\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Str

In [ ]:
import time


def create_chunks(documents):
    chunks = []
    successful = 0
    failed = 0
    
    for i, doc in enumerate(tqdm(documents)):
        try:
            doc_chunks = process_document(doc)
            if doc_chunks:
                chunks.extend(doc_chunks)
                successful += 1
            else:
                failed += 1
                
            
            if successful > 0 and successful % 5 == 0:
                print(f"\nProcessed {successful} documents. Sleeping for 30 seconds to avoid rate limits...")
                time.sleep(30)
                
        except Exception as e:
            print(f"Failed to process document {doc['source']}: {e}")
            failed += 1
            continue
    
    print(f" Done! Successfully processed {successful} documents, {failed} failed")
    return chunks

In [33]:
chunks = create_chunks(documents)

  0%|          | 0/76 [00:00<?, ?it/s]

  5%|▌         | 4/76 [00:11<03:46,  3.15s/it]


Processed 5 documents. Sleeping for 30 seconds to avoid rate limits...


  8%|▊         | 6/76 [01:26<27:33, 23.63s/it]


Processed 10 documents. Sleeping for 30 seconds to avoid rate limits...


...


Processed 70 documents. Sleeping for 30 seconds to avoid rate limits...


 97%|█████████▋| 74/76 [11:08<00:09,  4.96s/it]


Processed 75 documents. Sleeping for 30 seconds to avoid rate limits...


100%|██████████| 76/76 [11:47<00:00,  9.31s/it]


✅ Done! Successfully processed 76 documents, 0 failed


In [34]:
print(len(chunks))

603


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [35]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    # Generate embeddings using all-MiniLM-L6-v2 in batches to avoid memory issues
    batch_size = 32
    all_vectors = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        embeddings = embedding_model.encode(batch)
        all_vectors.extend(embeddings.tolist())
    
    vectors = all_vectors

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [36]:
create_embeddings(chunks)

Vectorstore created with 603 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [37]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [38]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [39]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [40]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [43]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with a valid JSON object containing a list of ranked chunk ids, nothing else. Format: {"order": [1, 2, 3, ...]}
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with a valid JSON object containing the list of ranked chunk ids."
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    try:
        response = completion(model=MODEL, messages=messages, response_format=RankOrder)
        reply = response.choices[0].message.content
        order = RankOrder.model_validate_json(reply).order
    except:
        # Fallback: try without response_format
        response = completion(model=MODEL, messages=messages)
        reply = response.choices[0].message.content
        json_match = re.search(r'\{.*\}', reply, re.DOTALL)
        if json_match:
            order = RankOrder.model_validate_json(json_match.group()).order
        else:
            print("Could not parse rerank response, using original order")
            order = list(range(1, len(chunks) + 1))
    
    print(order)
    return [chunks[i - 1] for i in order]

In [44]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    # Generate embedding using all-MiniLM-L6-v2
    query_embedding = embedding_model.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [45]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [47]:
for chunk in chunks:
    print(chunk.page_content[:30]+"...")

Annual Performance History

Th...
Annual Performance History

Ma...
Annual Performance History

Ra...
Annual Performance History

Ja...
Senior Data Engineer and Innov...
Annual Performance History

Th...
Annual Performance History

Th...
Annual Performance History

Th...
Performance History Continued
...
Additional Performance History...


In [48]:
reranked = rerank(question, chunks)

[5, 4, 3, 2, 6, 1, 10, 8, 7, 9]


In [52]:
for chunk in reranked:
    print(chunk.page_content[:]+"...")

Senior Data Engineer and Innovator

This section describes Maxine Thompson's promotion to Senior Data Engineer and her achievements, including leading a pivotal project and receiving the Insurellm Innovator of the year award.

- **January 2021 - Present**: **Senior Data Engineer**
  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award....
Annual Performance History

James Wilson's annual performance history at Insurellm.

## Annual Performance History
- **2023:** Rating: 4.9/5
  *Outstanding performance. Successfully led AI transformation initiative. Exceptional strategic leadership and team building. Key driver of company growth.*

- **2022:** Rating: 4.7/5
  *Exceed

In [50]:
chunks[6]

Result(page_content="Annual Performance History\n\nThis section provides an overview of Maxine Thompson's annual performance history, including her performance ratings and feedback.\n\n## Annual Performance History\n- **2017**: *Meets Expectations*  \n  Maxine showed potential in her role but struggled with initial project deadlines. Her adaptability and willingness to learn made positive impacts on her team.", metadata={'type': 'employees', 'source': 'knowledge-base/employees/Maxine Thompson.md'})

In [56]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

16


In [57]:
reranked = rerank(question, chunks)

[17, 3, 4, 5, 7, 6, 10, 13, 12, 9, 2, 1, 14, 8, 15, 16, 18, 19, 20]


In [58]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [59]:
reranked[0].page_content

"Compensation and Other HR Notes\n\nThis chunk provides information on Jessica Liu's compensation history and other HR notes, including her education, skills, and professional development.\n\n## Compensation History\n- **2023:** Base Salary: $92,000 + Bonus: $6,000\n- **2022:** Base Salary: $85,000 + Bonus: $4,000\n- **2021:** Base Salary: $72,000 + Bonus: $2,000\n- **2020:** Base Salary: $68,000\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.\n- **Work Style:** Prefers remote work. Strong written communicator. Participates actively in team standups and planning sessions.\n- **Feedback:** Reliable developer with good attention to UI details. Improving technical decision-making skills. Would benefit from more owne

In [60]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [61]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [62]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [63]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    history_text = "\n".join([f"{msg['role']}: {msg['content']}" for msg in history]) if history else "No conversation history."
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history_text}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "user", "content": message}])
    return response.choices[0].message.content.strip()

In [68]:
rewrite_query("Who won the IIOTY award?", [])

'IIOTY award winner?'

In [69]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [70]:
answer_question("Who won the IIOTY award?", [])

Who won IIOTY award?
[5, 19, 4, 17, 2, 3, 9, 12, 16, 10, 1, 6, 15, 14, 8, 7, 11, 20, 18, 13]


('According to the provided information, Maxine Thompson was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.',
 [Result(page_content="Senior Data Engineer and Innovator\n\nThis section describes Maxine Thompson's promotion to Senior Data Engineer and her achievements, including leading a pivotal project and receiving the Insurellm Innovator of the year award.\n\n- **January 2021 - Present**: **Senior Data Engineer**\n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.", metadata={'type': 'employees', 'source': 'knowledge-base/employees/Maxine Thompson.md'}),
  Result(page_content="Compensation and Other HR Notes\n

In [72]:
answer_question("Who went to Manchester University?", [])

Who attended Manchester University?
[15, 6, 14, 13, 9, 18, 7, 20, 8, 11, 4, 3, 5, 2, 10, 12, 16, 17, 19, 1]


('According to the provided information, Jessica Liu attended the University of Manchester, where she earned a BS in Computer Science.',
 [Result(page_content="Compensation and Other HR Notes\n\nThis chunk provides information on Jessica Liu's compensation history and other HR notes, including her education, skills, and professional development.\n\n## Compensation History\n- **2023:** Base Salary: $92,000 + Bonus: $6,000\n- **2022:** Base Salary: $85,000 + Bonus: $4,000\n- **2021:** Base Salary: $72,000 + Bonus: $2,000\n- **2020:** Base Salary: $68,000\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.\n- **Work Style:** Prefers remote work. Strong written communicator. Participates actively in team standups and pla